# Entity Sentiment Data Pipeline

This notebook explains how the sentiment analyzer works in the WigglNet data engine.

<small>© WigglNet 2026</small>

## Overview

#### What is a Sentiment Analyzer?
A sentiment analyzer identifies **named entities** (organizations, people, products, locations) in article text and determines the **sentiment** expressed toward each entity. WigglNet uses a mixture of different analyzers to identify trends and popular products or categories in a market, but the most commonly used combination would be a sequence of the spaCy (Named Entity Recognition) followed by the RoBERTa sentiment classifiction model. This process returns confidence scores and mention counts which is visualized and passed along to the application. These models also process articles in the same way as the keyword extractor.

### Marketable Use Cases in Product Selection
- Track brand or product sentiment in news articles
- Monitor public perception of companies, products, or people
- Allow clients to analyze compeditor sentiment
- Identify trends in entity coverage (positive vs negative)

## Architecture

The sentiment analyzer uses a two-stage pipeline:

Text example: <i>"These white windows emanate an airy, feelgood atmosphere."</i>

1) Name Entity Recognition (spaCy)

    Important words are identified and extracted based on their relation to a specified entity for the model to label.

  `["white","windows","airy","feelgood"] `

2) Sentiment Classification

    A ranking or classification is placed on each word and it is given a confidence score.

`Results: [{entity: "Apple", sentiment: "positive", confidence: 0.89, ...}]`

### Why Two Models?

1. **spaCy** - Fast, rule-based + statistical NER
   - Identifies entities (WHO/WHAT the text is about)
   - Categorizes entity types (ORG, PERSON, PRODUCT, etc.)
   - Processes entire documents in milliseconds

2. **RoBERTa** - Deep learning sentiment classifier
   - Understands contextual sentiment (HOW entities are discussed)
   - Pre-trained on Twitter data for robust social sentiment
   - Returns confidence scores for positive/neutral/negative

## Setup and Imports

First, let's import the sentiment analyzer and ensure models are downloaded.

In [1]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from app.keywords.tools.sentiment_analyzer import (
    ensure_models,
    analyze_entity_sentiment,
    add_sentiment_to_articles
)

# Download models if needed (only runs once)
print("Ensuring models are downloaded...")
ensure_models()
print("✓ Models ready!")

Ensuring models are downloaded...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 27549.92it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Models ready!


## Stage 1: Named Entity Recognition (spaCy)

spaCy identifies and categorizes entities in text.

### Entity Types Tracked

| Type | Description | Example |
|------|-------------|----------|
| ORG | Organizations | Apartment Therapy, Remodelista, UN |
| PERSON | Emily Henderson | Martha Stewert, Elon Musk |
| PRODUCT | Products/Services | Dining Table, Loveseat |
| GPE | Geopolitical entities | USA, California, Paris |
| NORP | Nationalities/groups | American, Republican |

Let's see spaCy NER in action:

In [2]:
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_md")

sample_text = """
Apple reported record earnings this quarter, exceeding analyst expectations.
Tim Cook credited strong iPhone sales and the growing services division.
Meanwhile, Microsoft announced sweeping layoffs affecting thousands of employees.
"""

doc = nlp(sample_text)

print("Entities found by spaCy:\n")
for ent in doc.ents:
    print(f"  {ent.text:<20} → {ent.label_:<10} ({spacy.explain(ent.label_)})")

Entities found by spaCy:

  Apple                → ORG        (Companies, agencies, institutions, etc.)
  this quarter         → DATE       (Absolute or relative dates or periods)
  Tim Cook             → PERSON     (People, including fictional)
  iPhone               → GPE        (Countries, cities, states)
  Microsoft            → ORG        (Companies, agencies, institutions, etc.)
  thousands            → CARDINAL   (Numerals that do not fall under another type)


## Stage 2: Sentiment Classification (RoBERTa)

RoBERTa analyzes the emotional tone of sentences containing entities.

### How RoBERTa Works

1. **Tokenization** - Breaks text into subword tokens
2. **Contextual Embeddings** - Converts tokens to vectors that capture meaning
3. **Classification** - Predicts sentiment label with confidence score

### Model Details
- **Model**: `cardiffnlp/twitter-roberta-base-sentiment-latest`
- **Training**: Fine-tuned on Twitter data (~124M tweets)
- **Output**: 3 classes (positive, neutral, negative) + confidence (0-1)

Let's test RoBERTa on individual sentences:

In [3]:
from transformers import pipeline

# Load RoBERTa sentiment classifier
classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True
)

sentences = [
    "Apple reported record earnings this quarter.",
    "Microsoft announced sweeping layoffs.",
    "The company continues to operate normally."
]

print("RoBERTa Sentiment Analysis:\n")
for sentence in sentences:
    result = classifier(sentence)[0]
    label = result['label']
    score = result['score']
    
    # Map LABEL_0/1/2 to readable names
    sentiment_map = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}
    sentiment = sentiment_map.get(label, label)
    
    print(f"  Sentence: {sentence}")
    print(f"  → {sentiment.upper()} (confidence: {score:.3f})\n")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 24995.70it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RoBERTa Sentiment Analysis:

  Sentence: Apple reported record earnings this quarter.
  → POSITIVE (confidence: 0.961)

  Sentence: Microsoft announced sweeping layoffs.
  → NEGATIVE (confidence: 0.690)

  Sentence: The company continues to operate normally.
  → NEUTRAL (confidence: 0.710)



## The Complete Pipeline

Now let's see how the two stages work together to analyze entity-level sentiment.

### Algorithm Steps

1. **Extract entities** - Use spaCy to find all named entities
2. **Tokenize sentences** - Split text into sentences
3. **Match entities to sentences** - Find which sentences mention each entity
4. **Classify sentiment** - Run RoBERTa on sentences containing the entity
5. **Aggregate scores** - Average confidence scores per entity
6. **Rank by mentions** - Sort entities by how often they appear

### Visual Example

```
Text: "Apple reported record earnings. Tim Cook praised the team."

Step 1 (spaCy NER):
  Entities: ["Apple", "Tim Cook"]

Step 2 (Tokenize):
  Sentences: [
    "Apple reported record earnings.",
    "Tim Cook praised the team."
  ]

Step 3 (Match):
  "Apple" → ["Apple reported record earnings."]
  "Tim Cook" → ["Tim Cook praised the team."]

Step 4 (RoBERTa):
  "Apple reported record earnings." → positive (0.89)
  "Tim Cook praised the team." → positive (0.92)

Step 5 (Results):
  Apple: positive, confidence=0.89, mentions=1
  Tim Cook: positive, confidence=0.92, mentions=1
```

## Loading Real Article Data

Using actual files from the /data folder to demonstrate sentiment analysis on real articles.

In [4]:
import json
import glob

# Find any JSON files in the data folder
data_pattern = '../app/keywords/data/*.json'
data_files = glob.glob(data_pattern)

if not data_files:
    raise FileNotFoundError(f"No JSON files found matching pattern: {data_pattern}")

# Use the most recent file (sorted by filename, which includes timestamp)
data_file = sorted(data_files)[-1]

with open(data_file, 'r') as f:
    articles = json.load(f)

print(f"Loaded {len(articles)} articles from {data_file}")
print(f"\nFirst article title: {articles[0]['title']}")
print(f"Number of paragraphs: {len(articles[0]['paragraphs'])}")

Loaded 146 articles from ../app/keywords/data/html_articles_20260404_235013.json

First article title: Room Ideas
Number of paragraphs: 6


## Full Example

Let's analyze a real article from our dataset:

In [5]:
# Use a real article from the dataset
sample_article = articles[0]
article_text = ' '.join(sample_article['paragraphs'])

print(f"Analyzing article: {sample_article['title']}\n")
print(f"Article length: {len(article_text)} characters")
print(f"Number of paragraphs: {len(sample_article['paragraphs'])}\n")

# Run the complete analysis
results = analyze_entity_sentiment(article_text, top_n=10)

print(f"Found {len(results)} entities with sentiment:\n")
print(f"{'Entity':<20} {'Type':<10} {'Sentiment':<12} {'Confidence':<12} {'Mentions'}")
print("="*70)

for r in results:
    print(
        f"{r['entity']:<20} {r['entity_type']:<10} {r['sentiment']:<12} "
        f"{r['confidence']:<12.3f} {r['mentions']}"
    )

Analyzing article: Room Ideas

Article length: 323 characters
Number of paragraphs: 6



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 29998.76it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Found 1 entities with sentiment:

Entity               Type       Sentiment    Confidence   Mentions
Hearst Digital Media We ORG        neutral      0.842        1


## Understanding the Results

### Result Fields

| Field | Type | Description |
|-------|------|-------------|
| `entity` | str | The entity name as it appears in text |
| `entity_type` | str | spaCy category (ORG, PERSON, etc.) |
| `sentiment` | str | Dominant sentiment (positive/neutral/negative) |
| `confidence` | float | RoBERTa confidence score (0-1) |
| `mentions` | int | Number of sentences mentioning this entity |

### Interpreting Confidence Scores

- **0.90 - 1.00**: Very confident - clear sentiment
- **0.70 - 0.89**: Confident - reliable sentiment
- **0.50 - 0.69**: Moderate - some uncertainty
- **< 0.50**: Low confidence - mixed or unclear sentiment

## Integration with Article Pipeline

The sentiment analyzer works with the same data format as the keyword extractor:

In [6]:
# Use real articles from the loaded data
# Select first two articles from the dataset
real_articles = articles[:2] if len(articles) >= 2 else articles[:1]

# Add sentiment analysis to the articles
articles_with_sentiment = add_sentiment_to_articles(real_articles, top_n=5, verbose=True)

# View results
print("\n" + "="*70)
for article in articles_with_sentiment:
    print(f"\nArticle: {article['title']}")
    if article.get('entity_sentiment'):
        for entity in article['entity_sentiment']:
            print(f"  - {entity['entity']}: {entity['sentiment']} (conf: {entity['confidence']:.2f}, mentions: {entity['mentions']})")
    else:
        print("  No entities detected")


Analyzing entity sentiment for 2 articles...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 21568.68it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    [1] Top entities: Hearst Digital Media We (neutral)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 27995.45it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


    [2] Top entities: House (positive), Eisenhart (neutral), Alexandra Gater (positive)
✓ Sentiment analyzed for 2 articles


Article: Room Ideas
  - Hearst Digital Media We: neutral (conf: 0.84, mentions: 1)

Article: 45 Clever Studio Apartment Ideas That Designers Swear By
  - House: positive (conf: 0.68, mentions: 3)
  - Eisenhart: neutral (conf: 0.84, mentions: 2)
  - Alexandra Gater: positive (conf: 0.95, mentions: 2)
  - Ikea: neutral (conf: 0.71, mentions: 2)
  - Dan Mazzarini: positive (conf: 0.71, mentions: 2)


## Database Storage

Entity sentiment results are stored in the database alongside articles:

### Tables

1. **ENTITIES**
   - `ENTITY_ID` (primary key)
   - `ENTITY_TEXT` (e.g., "Apple")
   - `ENTITY_TYPE` (e.g., "ORG")

2. **ARTICLE_ENTITY_SENTIMENT**
   - `ARTICLE_ID` (foreign key)
   - `ENTITY_ID` (foreign key)
   - `SENTIMENT` (positive/neutral/negative)
   - `CONFIDENCE` (0-1)
   - `MENTIONS` (count)

This allows querying like:
- "Show all articles with negative sentiment about Apple"
- "Track sentiment trend for Microsoft over time"
- "Find most-mentioned entities this month"

## Performance Considerations

### Speed
- spaCy NER: ~100-500 articles/second
- RoBERTa classification: ~10-50 sentences/second (GPU can 10x this)
- Total pipeline: ~5-10 articles/second on CPU

### Memory
- spaCy model: ~100MB RAM
- RoBERTa model: ~500MB RAM
- Total footprint: ~600-800MB

### First Run
Models are downloaded automatically on first use:
- spaCy `en_core_web_md`: ~40MB download
- RoBERTa model: ~500MB download

### Optimization Tips
1. Process articles in batches
2. Use GPU if available (set `device=0` in pipeline)
3. Filter entities by type before sentiment analysis
4. Cache results in database to avoid reprocessing

## Limitations & Future Improvements

### Current Limitations

1. **Sarcasm/Irony**: RoBERTa can misinterpret sarcastic statements
2. **Entity Disambiguation**: "Apple" could mean company or fruit
3. **Complex Sentiments**: Nuanced opinions may be oversimplified
4. **English Only**: Current models only support English text

### Potential Improvements

1. **Aspect-based Sentiment**: Analyze specific aspects (price, quality, service)
2. **Temporal Tracking**: Monitor sentiment changes over time
3. **Emotion Detection**: Beyond pos/neg - detect joy, anger, fear, etc.
4. **Cross-lingual**: Add support for Spanish, French, etc.
5. **Custom Training**: Fine-tune on domain-specific data (e.g., home design articles)

## Summary

The sentiment analyzer combines two powerful NLP models:

1. **spaCy** identifies WHO/WHAT (entities)
2. **RoBERTa** determines HOW they're discussed (sentiment)

This entity-level approach provides granular insights that document-level sentiment misses.

### Key Takeaways
- ✅ Fast and accurate entity detection
- ✅ State-of-the-art sentiment classification
- ✅ Confidence scores for quality assessment
- ✅ Easy integration with article pipeline
- ✅ Database-ready output format

### Next Steps
- See `web_scraper.py` for integration example
- Check `db_uploader.py` for database storage
- Explore SQL queries for sentiment analysis over time

## Model Evaluation & Accuracy Metrics

To assess the model's performance, we'll evaluate it on a labeled test dataset and calculate standard classification metrics.

### Metrics Explained

| Metric | Description |
|--------|-------------|
| **Accuracy** | Overall percentage of correct predictions |
| **Precision** | Of all predicted positives, how many were correct? |
| **Recall** | Of all actual positives, how many did we find? |
| **F1-Score** | Harmonic mean of precision and recall |
| **Confusion Matrix** | Visual breakdown of correct/incorrect predictions |

In [7]:
# Create a labeled test dataset for evaluation
# Using diverse examples from home design, food, travel, and general topics

test_data = [
    # Positive examples
    ("The garden looks absolutely stunning with these new flowers.", "positive"),
    ("This restaurant serves the most delicious pasta I've ever tasted!", "positive"),
    ("The sunset view from this beach is breathtaking and peaceful.", "positive"),
    ("Her interior design skills transformed the entire living room beautifully.", "positive"),
    ("I'm thrilled with how the renovation turned out.", "positive"),
    ("The chef prepared an incredible meal that exceeded all expectations.", "positive"),
    ("This hiking trail offers magnificent views and wonderful wildlife.", "positive"),
    ("The hotel staff provided exceptional service throughout our stay.", "positive"),
    ("The gallery exhibition was inspiring and thought-provoking.", "positive"),
    ("Outstanding craftsmanship on this handmade furniture piece.", "positive"),
    
    # Negative examples
    ("The paint color looks terrible and clashes with everything.", "negative"),
    ("Awful dining experience with cold food and rude staff.", "negative"),
    ("Don't visit this tourist trap, complete waste of time and money.", "negative"),
    ("The room renovation was a disaster from start to finish.", "negative"),
    ("I regret booking this vacation rental, it was filthy.", "negative"),
    ("Horrible weather ruined our entire beach weekend.", "negative"),
    ("The museum failed to maintain its exhibits properly.", "negative"),
    ("Poor quality materials that started falling apart immediately.", "negative"),
    ("This walking tour was boring and poorly organized.", "negative"),
    ("Disastrous decorating choices that made the space look worse.", "negative"),
    
    # Neutral examples
    ("The store opens at nine o'clock on weekdays.", "neutral"),
    ("The recipe calls for three cups of flour.", "neutral"),
    ("The museum is located in the historic district downtown.", "neutral"),
    ("The paint is available in twelve different shades.", "neutral"),
    ("They announced new visiting hours starting next month.", "neutral"),
    ("The apartment has two bedrooms and one bathroom.", "neutral"),
    ("The restaurant serves lunch from eleven to three.", "neutral"),
    ("The fabric comes with care instructions and warranty information.", "neutral"),
    ("Standard delivery takes five to seven business days.", "neutral"),
    ("The furniture is made of oak wood and metal hardware.", "neutral"),
]

test_texts = [text for text, _ in test_data]
true_labels = [label for _, label in test_data]

print(f"Created test dataset with {len(test_data)} samples")
print(f"  - Positive: {true_labels.count('positive')}")
print(f"  - Negative: {true_labels.count('negative')}")
print(f"  - Neutral: {true_labels.count('neutral')}")

Created test dataset with 30 samples
  - Positive: 10
  - Negative: 10
  - Neutral: 10


In [8]:
# Get predictions from the RoBERTa model
predictions_raw = classifier(test_texts)

# Map LABEL_0/1/2 to readable sentiment labels
sentiment_map = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}
predicted_labels = [sentiment_map.get(pred['label'], pred['label']) for pred in predictions_raw]

# Display some sample predictions
print("Sample Predictions:\n")
print(f"{'Text':<60} {'True':<10} {'Predicted':<10} {'Match'}")
print("="*90)
for i in range(min(5, len(test_texts))):
    match = "✓" if true_labels[i] == predicted_labels[i] else "✗"
    text_preview = test_texts[i][:57] + "..." if len(test_texts[i]) > 60 else test_texts[i]
    print(f"{text_preview:<60} {true_labels[i]:<10} {predicted_labels[i]:<10} {match}")

Sample Predictions:

Text                                                         True       Predicted  Match
The garden looks absolutely stunning with these new flowers. positive   positive   ✓
This restaurant serves the most delicious pasta I've ever... positive   positive   ✓
The sunset view from this beach is breathtaking and peace... positive   positive   ✓
Her interior design skills transformed the entire living ... positive   positive   ✓
I'm thrilled with how the renovation turned out.             positive   positive   ✓


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Calculate overall accuracy
accuracy = accuracy_score(true_labels, predicted_labels)

# Calculate precision, recall, and F1 for each class
precision, recall, f1, support = precision_recall_fscore_support(
    true_labels, 
    predicted_labels, 
    labels=['positive', 'neutral', 'negative'],
    average=None
)

# Calculate macro averages (unweighted mean)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    true_labels, 
    predicted_labels, 
    average='macro'
)

print("="*70)
print("OVERALL METRICS")
print("="*70)
print(f"Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")
print(f"Macro-averaged Precision: {precision_macro:.3f}")
print(f"Macro-averaged Recall: {recall_macro:.3f}")
print(f"Macro-averaged F1-Score: {f1_macro:.3f}")

print("\n" + "="*70)
print("PER-CLASS METRICS")
print("="*70)
print(f"{'Class':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support'}")
print("-"*70)

labels_order = ['positive', 'neutral', 'negative']
for i, label in enumerate(labels_order):
    print(f"{label:<12} {precision[i]:<12.3f} {recall[i]:<12.3f} {f1[i]:<12.3f} {support[i]}")

ModuleNotFoundError: No module named 'sklearn'

### Confusion Matrix

The confusion matrix shows where the model makes correct and incorrect predictions. Each row represents the true class, and each column represents the predicted class.

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

# Calculate confusion matrix
labels_order = ['positive', 'neutral', 'negative']
cm = confusion_matrix(true_labels, predicted_labels, labels=labels_order)

# Print confusion matrix in a nice format
print("Confusion Matrix:")
print("="*60)
print(f"{'':>12} | {'Predicted →':<40}")
print(f"{'True ↓':<12} | {'positive':<12} {'neutral':<12} {'negative':<12}")
print("-"*60)

for i, true_label in enumerate(labels_order):
    row_str = f"{true_label:<12} |"
    for j in range(len(labels_order)):
        row_str += f" {cm[i][j]:<11}"
    print(row_str)

print("\n" + "="*60)
print("How to Read:")
print("  - Diagonal values (top-left to bottom-right) = correct predictions")
print("  - Off-diagonal values = misclassifications")
print("  - Example: If 'positive' row shows [8, 2, 0], the model:")
print("    - Correctly predicted 8 positive samples")
print("    - Incorrectly predicted 2 positive samples as neutral")
print("    - Incorrectly predicted 0 positive samples as negative")

### Detailed Classification Report

The classification report provides a comprehensive summary of model performance including all metrics per class.

In [ ]:
from sklearn.metrics import classification_report

# Generate detailed classification report
report = classification_report(
    true_labels, 
    predicted_labels, 
    labels=['positive', 'neutral', 'negative'],
    target_names=['positive', 'neutral', 'negative'],
    digits=3
)

print("Detailed Classification Report:")
print("="*70)
print(report)
print("\nKey Insights:")
print("  - Support: Number of samples for each class")
print("  - Precision: How many predicted positives were actually positive")
print("  - Recall: How many actual positives were correctly identified")
print("  - F1-Score: Balance between precision and recall")
print("  - Macro avg: Unweighted average across all classes")
print("  - Weighted avg: Average weighted by class support")

### Interpreting the Results

**What Makes a Good Score?**

| Metric | Excellent | Good | Fair | Poor |
|--------|-----------|------|------|------|
| Accuracy | > 0.90 | 0.80-0.90 | 0.70-0.80 | < 0.70 |
| Precision/Recall | > 0.90 | 0.80-0.90 | 0.70-0.80 | < 0.70 |
| F1-Score | > 0.90 | 0.80-0.90 | 0.70-0.80 | < 0.70 |

**Common Issues to Watch For:**

1. **Class Imbalance**: If one class has much lower precision/recall, the model may struggle with that sentiment
2. **Confusion Patterns**: Look at the confusion matrix to see which classes are confused with each other
3. **Neutral Challenges**: Neutral sentiment is often hardest to detect, as it lacks strong indicators

**Improving Model Performance:**

- **More Training Data**: Fine-tune on domain-specific labeled data
- **Better Threshold**: Adjust confidence thresholds for borderline predictions
- **Ensemble Methods**: Combine predictions from multiple models
- **Feature Engineering**: Add context-aware features or entity information